# 🎯 Day 29 — Mock Interview Practice — **FAANG / Top Product Company Level**

**Week 5 · Day 1 · Fri Nov 20, 2026** · Tags: **10 × [FAANG], 8 × [HARD]**

> Natural step-up from Day 28: Day 28 pushed streaming schema-drift, dynamic priority queues and LLM-powered triage — Day 29 now asks you to implement FAANG-scale algorithms (collaborative filtering, survival curves, Aho–Corasick, Hopcroft–Karp) directly in SQL/Python, and to reason about newer lakehouse surfaces (pandas-on-Spark, Spark Connect, metric views, multi-window SLOs) where the docs are thin and the traps are real.

| # | Category | Topic | Tag |
|---|---|---|---|
| 1 | SQL Advanced | Item-item collaborative filtering (cosine similarity matrix) | [FAANG] |
| 2 | SQL Advanced | Kaplan-Meier survival curve (censoring, product-limit estimator) | [FAANG] |
| 3 | Python | Aho-Corasick multi-pattern automaton (fail links, overlapping matches) | [FAANG] |
| 4 | Python | Hopcroft-Karp bipartite matching (layered augmenting paths) | [FAANG] |
| 5 | PySpark | `applyInPandasWithState` — stateful streaming threshold detector | [FAANG] |
| 6 | PySpark | pandas-on-Spark (`pandas_api`) internals traps | [HARD] |
| 7 | Spark Opt | Speculative execution — straggler diagnosis & duplicate-write safety | [HARD] |
| 8 | Spark Opt | Spark Connect architecture — what breaks, what ships over the wire | [HARD] |
| 9 | ETL | Streaming lakehouse backfill — dual streams, separate checkpoints | [HARD] |
| 10 | ETL | GenAI corpus prep — exact + fuzzy dedup, PII scrub, leak-proof splits | [FAANG] |
| 11 | ETL | Kafka consumer-group rebalance & seek-to-timestamp replay | [FAANG] |
| 12 | Data Modeling | Faceted multi-parent taxonomy bridge (double-count trap) | [HARD] |
| 13 | Data Modeling | Semantic-layer governance with Unity Catalog metric views | [FAANG] |
| 14 | Snowflake | GEOGRAPHY vs GEOMETRY types, geofencing with search optimization | [HARD] |
| 15 | Snowflake | Cross-cloud replication, failover groups & lag-based failback | [HARD] |
| 16 | Monitoring | Multi-window multi-burn-rate SLO alerting | [FAANG] |
| 17 | Monitoring | Multi-root-cause triage (simultaneous failures, shared victims) | [HARD] |
| 18 | Monitoring | Predictive SLA ETA — regression forecast with p90 band | [FAANG] |

**How to practice**: spend up to 30 minutes per question on the markdown problem statement before looking at the solution. The solution cells are runnable and end with self-verifying asserts — run them after your attempt.

---

In [0]:
# Day 29 setup — datasets, imports, temp directories (shared by all questions)
import os, sys, math, time, uuid, random, warnings, hashlib, re
from collections import deque, defaultdict
from datetime import datetime
import pandas as pd
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.streaming.state import GroupState, GroupStateTimeout

BASE = f"/Volumes/workspace/default/etl_test_volume/d29_{uuid.uuid4().hex[:8]}"
print("scratch dir:", BASE)
print("spark version:", spark.version, "| serverless Spark Connect session")

# ---- shared dataset 1: ratings for Q1 (item-item collaborative filtering) ----
ratings = spark.createDataFrame([
    (1, 101, 5.0), (1, 102, 3.0), (1, 103, 4.0),
    (2, 101, 2.0), (2, 102, 5.0), (2, 104, 4.0),
    (3, 101, 4.0), (3, 102, 1.0), (3, 103, 4.5),
    (4, 103, 5.0), (4, 104, 2.0),
    (5, 101, 1.0), (5, 104, 5.0),
], "user_id int, item_id int, rating double")

# ---- shared dataset 2: churn events for Q2 (Kaplan-Meier survival) ----
churn = spark.createDataFrame([
    (1, "trial", 10, True, "small"), (2, "trial", 12, True, "small"), (3, "trial", 5, False, "small"),
    (4, "trial", 20, True, "small"), (5, "trial", 9, True, "small"), (6, "trial", 3, False, "small"),
    (7, "trial", 6, True, "small"), (8, "trial", 20, False, "small"),
    (9, "trial", 11, True, "enterprise"), (10, "trial", 13, True, "enterprise"), (11, "trial", 4, False, "enterprise"),
    (12, "trial", 9, True, "enterprise"), (13, "trial", 17, False, "enterprise"), (14, "trial", 3, True, "enterprise"),
    (15, "trial", 20, False, "enterprise"), (16, "trial", 8, True, "enterprise"),
], "user_id int, cohort string, tenure_days int, churned boolean, plan string")

print("setup done — datasets: ratings(13), churn(17)")

## Q1 — SQL Advanced — Item-Item Collaborative Filtering `[FAANG]`

**Scenario (Netflix/Amazon-style recommender interview):** You have an explicit ratings table. Build an **item-item collaborative filtering** recommender **entirely in SQL** — no UDFs, no pandas: compute pairwise item similarities as **cosine similarity over full rating vectors** and score a target user's unseen items as the similarity-weighted average of the items they already rated.

**Requirements**

1. Compute pairwise cosine similarity for every item pair that shares at least one rater. **Trap:** the cosine must use norms over *all* of each item's ratings (full vectors), not just the co-rated subset — `sqrt(sum(r_i^2))` over the whole column, only the *dot product* restricted to common raters.
2. Score user 5's unseen items: predicted rating = Σ(sim(i,j)·r_uj) / Σ|sim(i,j)| over items j the user already rated, excluding items they already rated from the candidate set.
3. Output the top-2 recommendations with scores rounded to 3 decimals.

In [0]:
# Solution Q1 — SQL-only item-item CF
ratings.createOrReplaceTempView("ratings")

spark.sql("DROP VIEW IF EXISTS item_item_similarity")
spark.sql("""
CREATE OR REPLACE TEMP VIEW item_item_similarity AS
WITH co AS (  -- dot products over common raters only
  SELECT a.item_id AS i1, b.item_id AS i2,
         SUM(a.rating * b.rating) AS dot
  FROM ratings a JOIN ratings b
    ON a.user_id = b.user_id AND a.item_id < b.item_id
  GROUP BY a.item_id, b.item_id
),
norms AS (  -- FULL-vector norms: every rating of each item counts
  SELECT item_id, SQRT(SUM(rating * rating)) AS norm
  FROM ratings GROUP BY item_id
)
SELECT co.i1, co.i2, co.dot / (MAX(n1.norm) * MAX(n2.norm)) AS cosine
FROM co
JOIN norms n1 ON co.i1 = n1.item_id
JOIN norms n2 ON co.i2 = n2.item_id
GROUP BY co.i1, co.i2, co.dot
""")

sim = spark.sql("SELECT * FROM item_item_similarity").orderBy("i1", "i2")
sim.show()

# score user 5's UNSEEN items (101 and 104 are already rated -> candidates 102, 103)
target_user = 5
scored = spark.sql(f"""
WITH sims AS (
    SELECT i1 AS item_a, i2 AS item_b, cosine FROM item_item_similarity
    UNION ALL
    SELECT i2, i1, cosine FROM item_item_similarity
),
target AS (SELECT item_id, rating FROM ratings WHERE user_id = {target_user})
SELECT s.item_b AS item_id,
       SUM(s.cosine * t.rating) / SUM(ABS(s.cosine)) AS predicted_rating
FROM sims s
JOIN target t ON s.item_a = t.item_id
LEFT ANTI JOIN target seen ON s.item_b = seen.item_id  -- exclude already-rated items
GROUP BY s.item_b
ORDER BY predicted_rating DESC
""")
scored.show()

# ---- parity check against an independent Python reference implementation ----
pdf = ratings.toPandas()
items = sorted(pdf.item_id.unique())
vec = {i: dict(g[["user_id", "rating"]].values) for i, g in pdf.groupby("item_id")}
import numpy as np
ref_sim = {}
for i in items:
    for j in items:
        if i < j and set(vec[i]) & set(vec[j]):
            common = set(vec[i]) & set(vec[j])
            dot = sum(vec[i][u] * vec[j][u] for u in common)
            n1 = math.sqrt(sum(x * x for x in vec[i].values()))
            n2 = math.sqrt(sum(x * x for x in vec[j].values()))
            ref_sim[(i, j)] = dot / (n1 * n2)

sql_pairs = {(int(r.i1), int(r.i2)): round(float(r.cosine), 6) for r in sim.collect()}
for pair, v in ref_sim.items():
    assert abs(sql_pairs[pair] - round(v, 6)) < 1e-9, (pair, sql_pairs.get(pair), v)

rated = set(pdf[pdf.user_id == target_user].item_id)   # items the user already rated
candidates = [i for i in items if i not in rated]
ref_scores = {}
for c in candidates:
    num = den = 0.0
    for j in rated:
        s = ref_sim.get((min(c, j), max(c, j)))
        if s is not None:
            r_j = pdf[(pdf.user_id == target_user) & (pdf.item_id == j)].rating.iloc[0]
            num += s * r_j
            den += abs(s)
    ref_scores[c] = num / den

sql_scores = {int(r.item_id): round(float(r.predicted_rating), 6) for r in scored.collect()}
print("SQL scores:", sql_scores)
print("reference :", {k: round(v, 6) for k, v in ref_scores.items()})
assert sql_scores == {k: round(v, 6) for k, v in ref_scores.items()}
print("Q1 OK: SQL cosine matrix + predictions match the Python reference; candidates exclude rated items")

## Q2 — SQL Advanced — Kaplan-Meier Survival Curve `[FAANG]`

**Scenario (Meta/Netflix growth-analytics interview):** A free-trial cohort of 16 users; each row records tenure in days and whether the user churned (`True`) or was censored (`False` — still active or left for another reason). Compute the **Kaplan-Meier product-limit survival curve** `S(t)` entirely in SQL, then find the **median survival time** (first `t` where `S(t) ≤ 0.5`).

**Requirements**

1. At each distinct event (churn) time `t`: `d_t` = users churning at `t`, `n_t` = users **at risk** (tenure ≥ t, churned or censored — censored users count as at-risk until their tenure). **Trap:** a naive self-join on `event_time <= tenure` double-counts; the at-risk set must count each user once per event time.
2. `S(t) = ∏_{t_i ≤ t} (1 − d_i/n_i)` — express the running product with `LN()`/`EXP()` over a window.
3. Break the curve out by the `plan` dimension (small vs enterprise) and report both median survival times.

In [0]:
# Solution Q2 — product-limit estimator via LN/EXP window
churn.createOrReplaceTempView("churn")

km = spark.sql("""
WITH events AS (
  SELECT plan, tenure_days AS t, COUNT(*) AS d   -- churn events only define the curve
  FROM churn WHERE churned = true GROUP BY plan, tenure_days
),
at_risk AS (
  -- each user contributes to n_t at event time t iff their tenure >= t (counted ONCE)
  SELECT e.plan, e.t AS t, e.d AS d, COUNT(*) AS n
  FROM events e JOIN churn c ON c.plan = e.plan AND c.tenure_days >= e.t
  GROUP BY e.plan, e.t, e.d
)
SELECT plan, t, d, n,
       1 - d/n                                   AS s_t,
       EXP(SUM(LN(1 - d/n)) OVER (PARTITION BY plan ORDER BY t
             ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS survival
FROM at_risk
""")
km.orderBy("plan", "t").show()

median = spark.sql("""
WITH curve AS (
  WITH events AS (
    SELECT plan, tenure_days AS t, COUNT(*) AS d
    FROM churn WHERE churned = true GROUP BY plan, tenure_days
  ),
  at_risk AS (
    SELECT e.plan, e.t AS t, e.d AS d, COUNT(*) AS n
    FROM events e JOIN churn c ON c.plan = e.plan AND c.tenure_days >= e.t
    GROUP BY e.plan, e.t, e.d
  )
  SELECT plan, t,
         EXP(SUM(LN(1 - d/n)) OVER (PARTITION BY plan ORDER BY t
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)) AS survival
  FROM at_risk
)
SELECT plan, MIN(t) AS median_survival_days
FROM curve WHERE survival <= 0.5 + 1e-9 GROUP BY plan  -- epsilon: LN/EXP round-trip
-- can leave S(t)=0.5 at 0.5000000000000001 (1 ULP); a strict <= would skip the median
""")
median.show()

# ---- parity check: independent pandas reference ----
def km_reference(df):
    times = sorted(df[df.churned].tenure_days.unique())
    s, curve = 1.0, []
    for t in times:
        d = int(((df.tenure_days == t) & df.churned).sum())
        n = int((df.tenure_days >= t).sum())
        s *= (1 - d / n)
        curve.append((t, s))
    return curve

med_ref = {}
for plan, g in churn.toPandas().groupby("plan"):
    curve = km_reference(g)
    med = next((t for t, s in curve if s <= 0.5), None)
    sql_curve = km.where(F.col("plan") == plan).orderBy("t").toPandas()
    for (t_ref, s_ref), row in zip(curve, sql_curve.itertuples()):
        assert int(row.t) == t_ref and abs(float(row.survival) - s_ref) < 1e-9, (plan, t_ref)
    med_ref[plan] = med

sql_med = {r.plan: int(r.median_survival_days) for r in median.collect()}
print("SQL medians:", sql_med, "| reference medians:", med_ref)
assert sql_med == med_ref
print("Q2 OK: SQL Kaplan-Meier curve and medians match the pandas reference (censoring handled, no at-risk double count)")

## Q3 — Python — Aho-Corasick Multi-Pattern Automaton `[FAANG]`

**Scenario (Meta/Google content-moderation interview):** Scan millions of chat messages against a **growing blocklist of thousands of phrases**. Re-running `str.find` per phrase is O(n·k); **Aho-Corasick** builds one automaton and scans each message **once**, finding *all* occurrences of *all* patterns, including overlapping ones, via goto/fail/output links.

**Requirements**

1. Build the automaton from scratch: goto trie → BFS fail links → output links (fail-state outputs chained onto the node).
2. Verify against a naive `text.count(p)` reference — **including overlapping matches** (e.g. pattern `"aa"` in `"aaaa"` must yield 3).
3. Demonstrate the classic moderator trap: blocklist contains `"hate"` and `"speech"`, message contains `"hate speech"` — the *composite* policy term is found while scanning each pattern independently.
4. Stress-test on 200 random texts/patterns against the naive reference.

In [0]:
# Solution Q3 — goto/fail/output links + one-pass scan
class AhoCorasick:
    def __init__(self, patterns):
        self.patterns = list(patterns)
        self.goto = [{}]                     # state -> {char -> state}
        self.fail = [0]                      # state -> fallback state
        self.out = [[]]                      # state -> matched pattern indices
        for idx, p in enumerate(self.patterns):
            state = 0
            for ch in p:
                if ch not in self.goto[state]:
                    self.goto.append({}); self.fail.append(0); self.out.append([])
                    self.goto[state][ch] = len(self.goto) - 1
                state = self.goto[state][ch]
            self.out[state].append(idx)
        # BFS fail links (root's children fail to root)
        q = deque(self.goto[0].values())
        while q:
            state = q.popleft()
            for ch, nxt in self.goto[state].items():
                q.append(nxt)
                f = self.fail[state]
                while f and ch not in self.goto[f]:
                    f = self.fail[f]
                cand = self.goto[f].get(ch)      # None when the root itself has no 'ch'
                self.fail[nxt] = cand if cand is not None and cand != nxt else 0
                self.out[nxt] = self.out[nxt] + self.out[self.fail[nxt]]  # output links

    def scan(self, text):
        """Yield (end_index, pattern_index) for EVERY match, overlaps included."""
        state, hits = 0, []
        for i, ch in enumerate(text):
            while state and ch not in self.goto[state]:
                state = self.fail[state]
            state = self.goto[state].get(ch, 0)
            for idx in self.out[state]:
                hits.append((i - len(self.patterns[idx]) + 1, idx))
        return hits

patterns = ["hate", "speech", "aa", "abc", "bca"]
ac = AhoCorasick(patterns)

text = "hate speech aaaa abcabca"
hits = ac.scan(text)
found = sorted((patterns[i], pos) for pos, i in hits)
print("found (pattern, start):", found)

# naive reference incl. overlaps
naive = []
for i, p in enumerate(patterns):
    start = text.find(p)
    while start != -1:
        naive.append((p, start))
        start = text.find(p, start + 1)     # step 1 -> overlaps counted
assert sorted(found) == sorted(naive)

# overlap proof, derived from the data rather than hand-positioned asserts:
aa_hits  = sorted(pos for pat, pos in found if pat == "aa")
abc_hits = sorted(pos for pat, pos in found if pat == "abc")
bca_hits = sorted(pos for pat, pos in found if pat == "bca")
print("'aa' overlapping matches at:", aa_hits)
print("'abc' matches at:", abc_hits, "| 'bca' matches at:", bca_hits)
assert len(aa_hits) == 3                       # 4 consecutive a's -> 3 overlapping 'aa'
assert ("hate", 0) in found and ("speech", 5) in found
assert len(abc_hits) == 2 and len(bca_hits) == 2  # 'abcabca' overlaps both

# the moderator trap: the composite term is present because BOTH parts match
msg = "this post contains hate speech and nothing else"
assert {p for p in patterns if p in msg} >= {"hate", "speech"}

random.seed(42)                              # 200 stress rounds vs the naive scan
for _ in range(200):
    t = "".join(random.choice("abc") for _ in range(random.randint(5, 30)))
    pats = ["".join(random.choice("abc") for _ in range(random.randint(1, 3)))
            for _ in range(random.randint(1, 4))]
    a = AhoCorasick(pats)
    fast = sorted((pats[i], pos) for pos, i in a.scan(t))
    slow = []
    for p in pats:
        s = t.find(p)
        while s != -1:
            slow.append((p, s)); s = t.find(p, s + 1)
    assert fast == sorted(slow), (t, pats, fast, slow)
print("Q3 OK: automaton matches naive reference on overlaps and 200 random cases")

## Q4 — Python — Hopcroft-Karp Bipartite Matching `[FAANG]`

**Scenario (record-linkage / assignment problems at scale, asked at Google/Amazon):** Match millions of source records to at most as many target accounts — a **maximum bipartite matching**. Greedy first-fit can leave up to 2x fewer matches than optimal; Hungarian/KM is overkill for unweighted matching. **Hopcroft-Karp** finds a maximum matching in `O(E·√V)` via **layered augmenting paths** (BFS phase + DFS phase).

**Requirements**

1. Implement Hopcroft-Karp from scratch (BFS builds layered free-vertex distances; DFS finds augmenting paths along the layers).
2. Demonstrate the greedy trap: a 3×2 graph where greedy matches **1** but the optimum is **2**.
3. Stress-test against a brute-force reference on 60 random graphs, and show the greedy-vs-optimal gap on a 400×400 random graph.

In [0]:
# Solution Q4 — layered BFS/DFS augmenting paths, verified vs brute force
from collections import deque

def hopcroft_karp(adj):
    """adj: dict left -> list of right. Returns (matching size, {left: right})."""
    INF = float("inf")
    pair_u = {u: None for u in adj}
    pair_v = {}
    dist = {}
    def bfs():
        q = deque()
        for u in adj:                              # layer 0 = free left vertices
            if pair_u[u] is None:
                dist[u] = 0; q.append(u)
            else:
                dist[u] = INF
        found = False
        while q:
            u = q.popleft()
            for v in adj[u]:
                w = pair_v.get(v)
                if w is None:
                    found = True                  # reached a free right vertex
                elif dist[w] == INF:
                    dist[w] = dist[u] + 1; q.append(w)
        return found
    def dfs(u):
        for v in adj[u]:
            w = pair_v.get(v)
            if w is None or (dist[w] == dist[u] + 1 and dfs(w)):
                pair_u[u] = v; pair_v[v] = u       # augment along the layer
                return True
        dist[u] = INF
        return False
    matching = 0
    while bfs():                                    # one BFS+DFS phase = maximal set
        for u in adj:                               # of vertex-disjoint augmenting paths
            if pair_u[u] is None and dfs(u):
                matching += 1
    return matching, pair_u

def greedy_match(adj):
    used, m = set(), {}
    for u, vs in adj.items():                      # first-fit in left order
        for v in vs:
            if v not in used:
                used.add(v); m[u] = v; break
    return len(m), m

def brute_force(adj):
    best = 0
    lefts = list(adj)
    def rec(i, used, cnt):
        nonlocal best
        if i == len(lefts):
            best = max(best, cnt); return
        u = lefts[i]
        rec(i + 1, used, cnt)                      # leave u unmatched
        for v in adj[u]:
            if v not in used:
                used.add(v); rec(i + 1, used, cnt + 1); used.remove(v)
    rec(0, set(), 0)
    return best

# trap graph: greedy grabs a->1 first and blocks both b and c
adj = {"a": [1, 2], "b": [1], "c": [1]}
g_size, g_match = greedy_match(adj)
hk_size, hk_match = hopcroft_karp(adj)
opt = brute_force(adj)
print(f"trap graph: greedy={g_size} {g_match} | HK={hk_size} {hk_match} | optimum={opt}")
assert hk_size == opt == 2 and g_size == 1

random.seed(11)                                     # 60-graph stress vs brute force
for trial in range(60):
    nl, nr = random.randint(2, 6), random.randint(2, 6)
    g = {}
    for u in range(nl):
        vs = [v for v in range(nr) if random.random() < 0.5]
        if not vs: vs = [random.randrange(nr)]
        g[f"u{u}"] = vs
    hk, _ = hopcroft_karp(g)
    assert hk == brute_force(g), (trial, g, hk)

random.seed(3)                                      # scale gap for the talk track
big = {}
for u in range(400):
    big[f"u{u}"] = sorted({random.randrange(400) for _ in range(6)})
gs, _ = greedy_match(big)
hs, _ = hopcroft_karp(big)
print(f"400x400 random graph: greedy={gs} vs Hopcroft-Karp={hs} (gap {hs-gs})")
assert hs >= gs and hs > 300
print("Q4 OK: trap graph greedy 1 < HK 2 == optimum; 60 stress graphs == brute force; 400x400 gap shown")

## Q5 — PySpark — Stateful Streaming with `applyInPandasWithState` `[FAANG]`

**Scenario (Uber/IoT-scale fleet telemetry interview):** A device fleet streams temperature readings. Emit per-reading alert state transitions per device: `OK → WARN` on the first high reading, `WARN → ALARM` on the **second consecutive** high reading, reset to `OK` on a normal reading — with alert state that **survives across micro-batches** in Spark Structured Streaming state, keyed by device. This is the 2025-era pandas-state API (`applyInPandasWithState`), replacing the older arbitrary-state `mapGroupsWithState` for Python.

**Requirements**

1. Simulate a 2-year backfill: write 10 event files, stream them in Trigger.AvailableNow **one file per micro-batch** so you can prove state persists across batches.
2. Maintain per-device state `(consec_high_count, last_temp)` in the `GroupState` store — note the API traps: state values are **plain tuples** (no attribute access), and `state.get` **raises** when no state exists yet — guard with `state.exists`.
3. Emit a row per reading with the transition status, into a memory sink, then assert the full WARN→ALARM→OK lifecycle for a device whose readings span three separate batches.

In [0]:
# Solution Q5 — per-device alert state machine across micro-batches
from pyspark.sql.streaming.state import GroupState, GroupStateTimeout

HIGH = 80.0
state_schema = T.StructType([T.StructField("consec_high", T.IntegerType()),
                             T.StructField("last_temp", T.DoubleType())])
out_schema = T.StructType([
    T.StructField("device_id", T.StringType()), T.StructField("event_ts", T.TimestampType()),
    T.StructField("temp", T.DoubleType()), T.StructField("consec_high", T.IntegerType()),
    T.StructField("status", T.StringType()),
])

def alert_state_machine(key, pdfs, state):
    """key: device_id; pdfs: iterator of pandas DataFrames; state: GroupState."""
    import pandas as pd
    for pdf in pdfs:
        pdf = pdf.sort_values("event_ts")
        for row in pdf.itertuples(index=False):
            st = state.get if state.exists else None   # TRAP: .get raises STATE_NOT_EXISTS when unset
            consec = st[0] if st is not None else 0     # TRAP: value is a plain tuple, not an object
            consec = consec + 1 if row.temp > HIGH else 0
            state.update((consec, float(row.temp)))
            status = "ALARM" if consec >= 2 else ("WARN" if consec == 1 else "OK")
            yield pd.DataFrame([{   # generator-of-DataFrames output contract
                "device_id": row.device_id, "event_ts": row.event_ts,
                "temp": float(row.temp), "consec_high": consec, "status": status,
            }])

# --- simulate the backfill: 10 files, one event batch each, dev2 spans three files ---
telemetry_batches = [
    [("dev1", "2026-11-20 10:00:00", 30.0), ("dev2", "2026-11-20 10:00:00", 85.0)],
    [("dev3", "2026-11-20 10:00:00", 40.0)],
    [("dev1", "2026-11-20 10:05:00", 20.0), ("dev2", "2026-11-20 10:05:00", 90.0)],
    [("dev2", "2026-11-20 10:10:00", 25.0)],
    [("dev3", "2026-11-20 10:10:00", 88.0)],
    [("dev3", "2026-11-20 10:11:00", 92.0)],
]
src = f"{BASE}_q5_src_{uuid.uuid4().hex[:6]}"   # unique per cell-run: re-running stays clean
os.makedirs(src, exist_ok=True)
schema = T.StructType([
    T.StructField("device_id", T.StringType()),
    T.StructField("event_ts", T.TimestampType()),
    T.StructField("temp", T.DoubleType()),   # explicit float64 — avoids int/long parquet mismatch
])
for i, rows in enumerate(telemetry_batches):
    (pd.DataFrame(rows, columns=["device_id", "event_ts", "temp"])
       .astype({"event_ts": "datetime64[us]", "temp": "float64"})   # us (not ns): Spark parquet timestamps
       .to_parquet(f"{src}/batch_{i}.parquet"))

name = f"d29_q5_{uuid.uuid4().hex[:6]}"
q = (spark.readStream.schema(schema)
     .option("maxFilesPerTrigger", 1)          # one file per micro-batch: state must persist
     .parquet(src)
     .groupBy("device_id")
     .applyInPandasWithState(alert_state_machine, out_schema, state_schema,
                             outputMode="update", timeoutConf=GroupStateTimeout.NoTimeout)
     .writeStream.option("checkpointLocation", f"{BASE}_q5_ckpt_{uuid.uuid4().hex[:6]}")
     .outputMode("update").queryName(name).format("memory")
     .trigger(availableNow=True).start())
q.awaitTermination()

res = spark.table(name).orderBy("device_id", "event_ts").toPandas()   # memory sink: read via spark.table
print(res.to_string(index=False))
assert len(res) == 8

dev2 = res[res.device_id == "dev2"].sort_values("event_ts")
lifecycle = list(zip(dev2.event_ts.astype(str), dev2.consec_high, dev2.status))
print("dev2 lifecycle (spans 3 micro-batches):", lifecycle)
assert [("2026-11-20 10:00:00", 1, "WARN"), ("2026-11-20 10:05:00", 2, "ALARM"),
        ("2026-11-20 10:10:00", 0, "OK")] == lifecycle
assert list(res[res.device_id == "dev3"].status) == ["OK", "WARN", "ALARM"]
print("Q5 OK: GroupState persisted across micro-batches — dev2 WARN -> ALARM -> OK, dev3 OK -> WARN -> ALARM")

## Q6 — PySpark — pandas-on-Spark (`pandas_api`) Internals Traps `[HARD]`

**Scenario (migration interview):** A team migrating pandas pipelines to distributed Spark decides to "just call `.pandas_api()`" — the pandas-on-Spark API. It mostly works, then breaks in production with index errors, driver-collection memory pressure, and silently-dropped columns. Explain **why** each trap happens and fix the code.

**Requirements** — for each trap: reproduce it, explain the root cause, apply the fix:

1. `set_index(...)` then `to_spark()` silently **drops the index column**.
2. Iterating `.index` (or calling `dict(series)` on the result) raises `PandasNotImplementedError`.
3. `assign(some_col=[...])` with a raw list raises `TypeError`.
4. `to_pandas()` on a large frame is a **driver single-node collection** (only the advice-warning stands between you and an OOM).
5. The correct idiom: `assign(col=lambda d: ...)` and groupby aggregation parity with real pandas.

In [0]:
# Solution Q6 — five live traps, each explained
ps_df = spark.createDataFrame([
    ("u1", "2026-01", 120.0), ("u2", "2026-01", 80.0),
    ("u1", "2026-02", 150.0), ("u2", "2026-02", 90.0),
], "user string, month string, spend double").pandas_api()

# TRAP 1: set_index then to_spark() DROPS the index (it is not a real column)
idx_df = ps_df.set_index("user")
with warnings.catch_warnings(record=True) as w:
    warnings.simplefilter("always")
    back = idx_df.to_spark()
dropped = "user" not in back.columns
print(f"TRAP 1: set_index('user') -> to_spark() drops index? {dropped}; columns now {back.columns}")
print("        why: the index is a pandas-side overlay, not part of the Spark schema;")
print("        fix: reset_index() first, or keep the key as a regular column.")
assert dropped

# TRAP 2: iterating a pandas-on-Spark index raises PandasNotImplementedError
trap2 = None
try:
    list(idx_df.index)
    trap2 = "NO ERROR"
except Exception as e:
    trap2 = type(e).__name__
print(f"TRAP 2: iterating .index -> {trap2}")
print("        why: the index is lazily materialized; iteration is not implemented.")
assert trap2 == "PandasNotImplementedError"

# TRAP 3: assign with a raw list raises TypeError
trap3 = None
try:
    ps_df.assign(doubled=[1, 2, 3, 4])
    trap3 = "NO ERROR"
except Exception as e:
    trap3 = type(e).__name__
print(f"TRAP 3: assign(list) -> {trap3}")
print("        why: a raw python list has no distributed lineage;")
print("        fix: assign from an existing column with a lambda (stays distributed).")
assert trap3 == "TypeError"

# TRAP 4: to_pandas() is a full driver collection
with warnings.catch_warnings(record=True) as w2:
    warnings.simplefilter("always")
    pdf = ps_df.to_pandas()
print(f"TRAP 4: to_pandas() collected {len(pdf)} rows and raised {len(w2)} advice-warning(s) "
      f"(driver memory pressure at scale; fix: aggregate first, or use show()/collect sparingly)")

# TRAP 4b: even dict(ps_series) fails — it iterates the index internally
trap4b = None
try:
    dict(ps_df.groupby("user").spend.sum())
    trap4b = "NO ERROR"
except Exception as e:
    trap4b = type(e).__name__
print(f"TRAP 4b: dict(ps_series) -> {trap4b} (fix: .to_pandas().to_dict() on small results)")
assert trap4b == "PandasNotImplementedError"

# FIX idiom: lambda assign keeps the transformation in Spark
ok = ps_df.assign(doubled=lambda d: d.spend * 2)
g_ps = ok.groupby("user").doubled.sum().to_pandas().to_dict()
g_ref = pdf.assign(doubled=pdf.spend * 2).groupby("user").doubled.sum().to_dict()
print("groupby parity:", g_ps, "|", g_ref)
assert g_ps == g_ref
print("Q6 OK: 5 traps reproduced, explained, and fixed; groupby matches real pandas")

## Q7 — Spark Optimization — Speculative Execution `[HARD]`

**Scenario (production firefight interview):** A 40-minute stage always runs ~35 min fast, then hangs on **one straggler task** (a bad disk / noisy neighbor / GC-thrashing executor). Explain **speculative execution**: when Spark launches duplicate copies, how the winner is chosen, why it's safe (and when it is NOT safe), and the config knobs. Then quantify the win.

**Requirements**

1. Explain the mechanism: progress-based detection (`spark.speculation.quantile` task-progress threshold vs the median), re-launch on a free node, **first-committer-wins**, duplicate killed.
2. Safety analysis: why idempotent **output committers** are required — and the Delta Lake angle: duplicate writes to the same Delta table cause **optimistic-concurrency conflicts (ConcurrentAppendException)**, so speculative tasks must not both commit.
3. Simulate: stage with task durations `[10, 10, 10, 45]`; show makespan with and without speculation (duplicate launched at t=15 completes in 10 on a healthy node).

In [0]:
# Solution Q7 — mechanism, safety, and a quantified simulation

def run_stage(task_durations):
    """No speculation: stage completes when the slowest task finishes."""
    return max(task_durations)

def run_stage_speculative(task_durations, check_at=15, healthy_rewind=10):
    """Speculation: tasks still running at check_at that will overshoot the completed
    median by a wide factor get a duplicate; the FIRST finisher wins."""
    done = [d for d in task_durations if d <= check_at]
    med = sorted(done)[len(done) // 2]           # progress-threshold surrogate
    finish_times = []
    for d in task_durations:
        if d <= check_at:
            finish_times.append(d)               # finished before the check
        else:
            spec_finish = check_at + healthy_rewind   # duplicate on a healthy node
            finish_times.append(min(d, spec_finish))  # first-committer-wins
    return max(finish_times)

tasks = [10, 10, 10, 45]
no_spec = run_stage(tasks)
with_spec = run_stage_speculative(tasks)
print(f"stage tasks {tasks}: makespan without speculation = {no_spec} min, with = {with_spec} min")
assert with_spec == 25 and no_spec == 45
print("\nTalk track — configs and why:")
print("  spark.speculation=true              # master switch (off by default)")
print("  spark.speculation.multiplier=2.0     # task must be 2x SLOWER than the median")
print("  spark.speculation.quantile=0.75       # compare against the 75th-percentile progress")
print("  spark.speculation.interval=100ms     # scheduler re-check cadence")
print("  spark.speculation.minTaskRuntime=...  # don't speculate young tasks (false positives)")
print("\nSafety:")
print("  * File/HDFS sinks: committers make duplicates benign (first committer wins, other reverts).")
print("  * Delta sinks: duplicate tasks race to commit the same txn -> ConcurrentAppendException;")
print("    idempotent, append-only or retryable writes are required.")
print("  * Non-deterministic UDFs: duplicates may produce different rows -> must be deterministic.")
print("\nQ7 OK: speculation cut the stage 45 -> 25 min; first-committer-wins and Delta OCC caveats stated")

## Q8 — Spark Optimization — Spark Connect Internals `[HARD]`

**Scenario (platform-migration interview):** Notebooks and the serverless SQL warehouse now run **Spark Connect** (client–server Spark: a thin client ships a serialized *plan*, not code). A legacy job full of `spark.sparkContext`, `_jdf` JVM handles, `sc.parallelize`, and pandas conversions must be migrated. Explain the architecture and prove what breaks — live.

**Requirements**

1. Explain the architecture: client (gRPC) → serialized logical plan → server-side execution; where UDFs/lambdas ship (serialized closures, Arrow-based data channel); why the driver/client split changes everything.
2. Live-probe which classic APIs fail on Spark Connect — `sc.parallelize`, `df._jdf`, `spark._jsc` — and which still work (lazy DataFrame relations, `pandas_api`, UDFs).
3. State the migration playbook: RDD-free code, no JVM access, `spark.addArtifact` for modules/files instead of cluster-installed eggs, and why `toPandas()`/`collect()` are now *client-side* memory events.

In [0]:
# Solution Q8 — live probes of the client-server boundary
probes = {}

# 1: RDD API — no execution-context objects on the thin client
try:
    spark.sparkContext.parallelize([1, 2, 3])
    probes["sc.parallelize"] = "AVAILABLE"
except Exception as e:
    probes["sc.parallelize"] = f"{type(e).__name__}: {str(e)[:70]}"

# 2: JVM DataFrame handle — there is no local JVM
try:
    _ = spark.range(1)._jdf
    probes["df._jdf"] = "AVAILABLE"
except Exception as e:
    probes["df._jdf"] = type(e).__name__

# 3: spark context JVM — same reason
try:
    _ = spark._jsc
    probes["spark._jsc"] = "AVAILABLE"
except Exception as e:
    probes["spark._jsc"] = type(e).__name__

for k, v in probes.items():
    print(f"{k:16s} -> {v}")
assert all("AVAILABLE" not in v for v in probes.values())

# 4: lazy relations + UDFs still work — the plan ships, the lambda ships, data streams back via Arrow
@F.udf("double")
def plus_one(x):
    return None if x is None else float(x + 1)

r = spark.range(3).select(plus_one("id").alias("v")).toPandas()["v"].tolist()
print("UDF over Spark Connect:", r)
assert r == [1.0, 2.0, 3.0]

print("\nTalk track:")
print("* Client builds a LOGICAL PLAN; gRPC ships it; server plans + executes it.")
print("* Python lambdas/UDFs are serialized to the server (this is why closures must be picklable).")
print("* RDDs/JVM handles need a local SparkContext -> PySparkAttributeError on Connect.")
print("* Files/py modules: spark.addArtifact / %pip install on the SERVER, not client-side eggs.")
print("* collect()/toPandas() pull the result over gRPC to the CLIENT -> client memory, driver rules apply.")
print("* Migration: replace .rdd with DataFrame ops; replace _jvm hacks with native APIs;")
print("  replace sc.parallelize with spark.createDataFrame / spark.range.")
print("\nQ8 OK: classic-only APIs fail fast (PySparkAttributeError); relations + UDFs work")

## Q9 — ETL — Streaming Lakehouse Backfill (Dual Streams, Separate Checkpoints) `[HARD]`

**Scenario (migration-to-Lakeflow interview):** Migrate two years of history AND a live event stream into one Delta table **without stopping the live feed**. Two Structured Streaming queries: a **backfill stream** (history files, ordered) and a **live stream** (new files only), each with its **own checkpoint** — checkpoint offsets must never collide. Then survive two disasters: the backfill re-runs with a fresh checkpoint (crash recovery) and re-reads everything — the sink must absorb the duplicates.

**Requirements**

1. Two directories (history + live), one Delta target; `foreachBatch` + `MERGE` on `event_id` (idempotent upsert sink).
2. Prove: backfill stream loads 7 events, live stream adds 2, re-running the backfill with the **same** checkpoint re-reads nothing, and re-running with a **fresh** checkpoint (crash-recovery simulation) still ends with exactly 9 rows and **zero duplicate keys**.
3. Explain why `availableNow` + separate checkpoints per stream is the correct dual-stream pattern, and what would break if both streams shared one checkpoint location.

In [0]:
# Solution Q9 — two streams, two checkpoints, one idempotent MERGE sink
bf_dir = f"{BASE}_q9_backfill"; live_dir = f"{BASE}_q9_live"; target = f"{BASE}_q9_target"
os.makedirs(bf_dir, exist_ok=True); os.makedirs(live_dir, exist_ok=True)

history = [   # two years of events, one file per 'month'
    [("e1", "u1", "order", "2025-01-15"), ("e2", "u2", "order", "2025-02-20"), ("e3", "u1", "refund", "2025-03-05")],
    [("e4", "u3", "order", "2025-04-11"), ("e5", "u2", "order", "2025-05-30")],
    [("e6", "u4", "order", "2026-01-02"), ("e7", "u3", "refund", "2026-02-18")],
]
live_events = [[("e8", "u5", "order", "2026-11-18"), ("e9", "u1", "order", "2026-11-19")]]
cols = ["event_id", "user_id", "kind", "event_date"]
for i, rows in enumerate(history):
    pd.DataFrame(rows, columns=cols).to_parquet(f"{bf_dir}/hist_{i}.parquet")
for i, rows in enumerate(live_events):
    pd.DataFrame(rows, columns=cols).to_parquet(f"{live_dir}/live_{i}.parquet")

q9_schema = T.StructType([
    T.StructField("event_id", T.StringType()), T.StructField("user_id", T.StringType()),
    T.StructField("kind", T.StringType()), T.StructField("event_date", T.StringType()),
])
spark.createDataFrame([], q9_schema).write.format("delta").mode("overwrite").save(target)  # empty target

def make_upsert(tgt):
    def upsert(batch_df, batch_id):
        batch_df.createOrReplaceTempView("d29_q9_updates")
        batch_df.sparkSession.sql(f"""
        MERGE INTO delta.`{tgt}` t USING d29_q9_updates s
        ON t.event_id = s.event_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
        """)
    return upsert

def run_stream(src_dir, ckpt, name):
    q = (spark.readStream.schema(q9_schema).parquet(src_dir)
         .writeStream.foreachBatch(make_upsert(target))
         .option("checkpointLocation", ckpt)
         .queryName(name)
         .trigger(availableNow=True).start())
    q.awaitTermination()
    return spark.read.format("delta").load(target).count()

bf_ckpt  = f"{BASE}_q9_ckpt_bf_{uuid.uuid4().hex[:6]}"    # unique per cell-run
live_ckpt = f"{BASE}_q9_ckpt_live_{uuid.uuid4().hex[:6]}"
n0 = spark.read.format("delta").load(target).count()
print("target before:", n0); assert n0 == 0

n1 = run_stream(bf_dir, bf_ckpt, f"d29q9_bf_{uuid.uuid4().hex[:6]}")
print("after backfill stream:", n1); assert n1 == 7
n2 = run_stream(live_dir, live_ckpt, f"d29q9_live_{uuid.uuid4().hex[:6]}")
print("after live stream   :", n2); assert n2 == 9

n3 = run_stream(bf_dir, bf_ckpt, f"d29q9_bf2_{uuid.uuid4().hex[:6]}")  # SAME checkpoint
print("backfill re-run (same checkpoint):", n3); assert n3 == 9

n4 = run_stream(bf_dir, f"{BASE}_q9_ckpt_bf_replay_{uuid.uuid4().hex[:6]}",   # FRESH checkpoint
                f"d29q9_bf3_{uuid.uuid4().hex[:6]}")
print("backfill replay (fresh checkpoint, MERGE dedupes):", n4); assert n4 == 9

tgt_df = spark.read.format("delta").load(target)
dups = tgt_df.groupBy("event_id").count().filter("count > 1").count()
total = tgt_df.count()
print(f"final: {total} rows, duplicate event_ids: {dups}")
assert total == 9 and dups == 0
print("\nTalk track:")
print("* availableNow processes everything pending then stops - perfect for bounded backfills.")
print("* ONE checkpoint per stream: the file-source journal in a shared checkpoint would be")
print("  OVERWRITTEN by whichever stream starts first -> silent data loss.")
print("* MERGE-on-natural-key makes every re-run idempotent: crash recovery replays batches,")
print("  duplicates collapse (idempotent upsert), exactly-once effect at the sink.")
print("\nQ9 OK: dual streams, separate checkpoints, replay-safe, 0 duplicates")

## Q10 — ETL — GenAI Corpus Prep (Dedup, PII, Leak-Proof Splits) `[FAANG]`

**Scenario (LLM fine-tuning/RAG pipeline interview):** Before training or grounding a model on company documents you must build a **clean corpus**: exact + near-duplicate removal (memorization and benchmark contamination), **PII scrubbing**, token budget accounting, and a **leak-proof train/val split** where near-duplicates never straddle splits. Build the whole prep stage.

**Requirements**

1. **Exact dedup** by normalized content hash (whitespace/case-normalized SHA-256).
2. **Fuzzy dedup**: 3-gram **shingle Jaccard ≥ 0.7** = near-duplicate (the industry-standard approach — also mention MinHash for scale).
3. **PII scrub**: redact emails and 16-digit card numbers with regex.
4. **Leak-proof split**: bucket each *document* (not each token) by hash into train/val so near-dups land in the same split; report token counts.
5. Assert: 6 docs → 4 after dedup, 3 PII entities redacted, splits partition the corpus with no overlap.

In [0]:
# Solution Q10 — dedup -> scrub -> split pipeline
docs = {
    "doc1": "Alice's email is alice@example.com and her card is 4111-1111-1111-1111. She lives in Austin.",
    "doc2": "Bob reached out at bob@example.org about the new data platform launch in November.",
    "doc3": "Bob reached out at bob@example.org about the new data platform launch in November.",   # exact dup
    "doc4": "Bob reached out at bob@example.org regarding the new data platform launch in November.",  # near dup (1 word changed)
    "doc5": "Quarterly revenue grew 12% driven by the enterprise segment and renewals.",
    "doc6": "The engineering team shipped the streaming pipeline to production on Friday.",
}

def normalize(t): return re.sub(r"\s+", " ", t.strip().lower())
def shingles(t, k=3): return {normalize(t)[i:i+k] for i in range(len(normalize(t)) - k + 1)}
def jaccard(a, b):
    A, B = shingles(a), shingles(b)
    u = len(A | B)
    return len(A & B) / u if u else 0.0

# 1) exact dedup by content hash
seen_hash, exact_dups = {}, []
for k, v in docs.items():
    h = hashlib.sha256(normalize(v).encode()).hexdigest()
    if h in seen_hash:
        exact_dups.append(k)
    else:
        seen_hash[h] = k
print("exact duplicates removed:", exact_dups)
assert exact_dups == ["doc3"]

# 2) fuzzy dedup on the survivors (k-shingle Jaccard >= 0.7)
kept = [k for k in docs if k not in exact_dups]
pairs = [(kept[i], kept[j], round(jaccard(docs[kept[i]], docs[kept[j]]), 3))
         for i in range(len(kept)) for j in range(i + 1, len(kept))]
best = max(pairs, key=lambda x: x[2])
print("highest-jaccard pair:", best)
assert best[2] >= 0.7 and {best[0], best[1]} == {"doc2", "doc4"}
final = [k for k in kept if k != "doc4"]
print(f"corpus after fuzzy dedup: {final}  ({len(docs)} -> {len(final)} docs)")
assert len(final) == 4

# 3) PII scrub
PII_RULES = {
    "email": re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+"),
    "card":  re.compile(r"\b\d{4}-\d{4}-\d{4}-\d{4}\b"),
}
def scrub(t):
    found = []
    for kind, rx in PII_RULES.items():
        found += [kind] * len(rx.findall(t))
        t = rx.sub(f"<{kind.upper()}>", t)
    return t, found
scrubbed, total_pii = {}, 0
for k in final:
    s, f = scrub(docs[k])
    scrubbed[k] = s; total_pii += len(f)
print(f"PII entities redacted: {total_pii}")
assert total_pii == 3 and "alice@example.com" not in " ".join(scrubbed.values())

# 4) leak-proof split: bucket by DOCUMENT hash so near-dups share a split
def bucket(k, nbuckets=2): return int(hashlib.md5(k.encode()).hexdigest(), 16) % nbuckets
splits = {"train": [], "val": []}
for k in scrubbed:
    splits["val" if bucket(k) == 0 else "train"].append(k)
print("leak-proof split:", splits)
assert set(splits["train"]) | set(splits["val"]) == set(final)
assert not set(splits["train"]) & set(splits["val"])

ntok = {k: len(scrubbed[k].split()) for k in scrubbed}
print("token counts (whitespace, pre-tokenizer surrogate):", ntok)
print("\nTalk track: at scale swap the O(n^2) Jaccard for MinHash+LSH banding;")
print("swap regex for NER-based scrubbing; document-level (not row-level) bucketing is what")
print("prevents benchmark contamination in eval splits.")
print("\nQ10 OK: 6 -> 4 docs, 3 PII redactions, hash-bucketed leak-proof split")

## Q11 — ETL — Kafka Consumer-Group Rebalance & Seek-to-Timestamp Replay `[FAANG]`

**Scenario (streaming-platform interview):** During a consumer-group rebalance, partitions are revoked and reassigned mid-processing; on crash, **committed offsets lag behind processed records** (at-least-once) and the replay path is `seekToTimestamp` re-processing. Design the consumer + sink so that rebalances, crash redelivery, and replays are all safe — and simulate all three events.

**Requirements**

1. Simulate a consumer group over 4 partitions: eager (stop-the-world) rebalance when a 3rd consumer joins — revoke-all, then reassign; committed offsets survive in `__consumer_offsets`.
2. Simulate the crash window: records processed but offsets NOT committed → after "recovery" they are **redelivered**; prove the sink stays consistent.
3. Simulate `seekToTimestamp` replay of the full topic; prove no double-counting and explain why `enable.auto.commit=false` + idempotent upsert-by-key is the correct pattern.

In [0]:
# Solution Q11 — rebalance / crash / replay simulation with an idempotent sink
class FakeTopic:
    def __init__(self, records):          # (offset, partition, key, value)
        self.records = records
    def partition_records(self, p):
        return [r for r in self.records if r[1] == p]

class ConsumerGroup:
    """Eager (stop-the-world) rebalance: all partitions revoked, then reassigned round-robin."""
    def __init__(self, topic, n_partitions):
        self.topic, self.np = topic, n_partitions
        self.assignment = {}
        self.committed = {p: 0 for p in range(n_partitions)}   # __consumer_offsets
        self.processed = {}                                     # idempotent upsert sink
    def rebalance(self, members):
        print(f"  [rebalance] revoke-all: {self.assignment or '(empty)'} -> assign round-robin over {members}")
        self.assignment = {m: [] for m in members}
        for i, p in enumerate(range(self.np)):
            self.assignment[members[i % len(members)]].append(p)
        print(f"  [rebalance] new assignment: {self.assignment}")
    def consume(self, consumer):
        for p in self.assignment[consumer]:
            for off, part, key, val in self.topic.partition_records(p):
                if off >= self.committed[p]:
                    self.processed[key] = val                     # last-write-wins upsert
                    self.committed[p] = off + 1                    # auto-commit OFF: commit post-process
    def replay_from_timestamp(self, ts_cutoff):
        for p in range(self.np):                                  # seekToTimestamp(0)
            self.committed[p] = 0

records = [(i, i % 4, f"k{i}", f"v{i}") for i in range(12)]
cg = ConsumerGroup(FakeTopic(records), 4)

print("1) initial group of 2 consumers:")
cg.rebalance(["c1", "c2"])
cg.consume("c1"); cg.consume("c2")
print("  processed:", len(cg.processed), "keys")
assert len(cg.processed) == 12

print("2) c3 joins -> eager rebalance (offsets survive, no re-delivery):")
cg.rebalance(["c1", "c2", "c3"])
sizes = sorted(len(v) for v in cg.assignment.values())
assert sizes == [1, 1, 2]                                       # 4 partitions / 3 consumers
cg.consume("c1"); cg.consume("c2"); cg.consume("c3")
print("  processed after reconsume:", len(cg.processed))
assert len(cg.processed) == 12

print("3) crash before offset commit on partition 0 -> redelivery absorbed:")
cg.committed[0] = 0                                             # records 'processed, not committed'
cg.consume("c1")
print("  processed after redelivery:", len(cg.processed), "(idempotent upsert: no double-count)")
assert len(cg.processed) == 12

print("4) full seekToTimestamp replay:")
cg.replay_from_timestamp(0)
cg.consume("c1"); cg.consume("c2"); cg.consume("c3")
print("  processed after replay:", len(cg.processed))
assert len(cg.processed) == 12
print("\nTalk track: eager rebalances stop the world; cooperative (incremental) rebalancing")
print("(KIP-429) shrinks the pause. enable.auto.commit=false + commit-after-upsert gives")
print("at-least-once delivery; the MERGE/upsert-by-key sink upgrades it to exactly-once EFFECT.")
print("\nQ11 OK: rebalance, crash redelivery, and replay all safe; 12 keys throughout")

## Q12 — Data Modeling — Faceted Multi-Parent Taxonomy Bridge `[HARD]`

**Scenario (e-commerce analytics interview):** Products belong to a **category tree stored as an ancestor-closure bridge** (each product links to its leaf node AND every ancestor — the standard Kimball bridge for ragged/deep hierarchies). A new analyst rolls revenue up by joining the fact table to the bridge and summing every node; the numbers are inflated. Diagnose and fix.

**Requirements**

1. Show the trap: revenue by node from the raw join, and the grand total that results (**1800 vs the true 900**).
2. Explain why the ancestor closure double-counts (leaf + ancestors each carry the full revenue).
3. Fix it **in SQL**: aggregate at leaf level first, then roll up ancestors from leaf aggregates — every node's revenue must equal what the fact table says, and the parent must equal the sum of its leaves.

In [0]:
# Solution Q12 — ancestor-closure trap and the leaf-first rollup
products = spark.createDataFrame([
    ("p1", "iPhone 15", 300.0), ("p2", "Pixel 8", 200.0),
    ("p3", "MacBook Air", 150.0), ("p4", "ThinkPad", 150.0),
    ("p5", "iPad", 100.0),
], "product_id string, product_name string, revenue double")
products.createOrReplaceTempView("products")

bridge = spark.createDataFrame([   # leaf + ancestor rows (closure)
    ("p1", "Phones"), ("p1", "Electronics"),
    ("p2", "Phones"), ("p2", "Electronics"),
    ("p3", "Laptops"), ("p3", "Electronics"),
    ("p4", "Laptops"), ("p4", "Electronics"),
    ("p5", "Tablets"), ("p5", "Electronics"),
], "product_id string, node string")
bridge.createOrReplaceTempView("bridge")

fact_total = products.agg(F.sum("revenue")).first()[0]
print("fact-level total revenue:", fact_total)

naive = spark.sql("""
SELECT b.node, SUM(p.revenue) AS revenue, COUNT(DISTINCT b.product_id) AS n_products
FROM bridge b JOIN products p ON b.product_id = p.product_id
GROUP BY b.node ORDER BY revenue DESC
""").toPandas()
print("naive node rollup (whole-closure join):")
print(naive.to_string(index=False))
naive_total = naive.revenue.sum()
print(f"naive grand total across nodes: {naive_total}  <- 2x the true {fact_total}")
assert naive_total > fact_total

fixed = spark.sql("""
WITH leaf_nodes AS (
  SELECT b.node AS leaf, SUM(p.revenue) AS leaf_revenue,
         COUNT(DISTINCT b.product_id) AS n_products
  FROM bridge b JOIN products p ON b.product_id = p.product_id
  WHERE b.node != 'Electronics'      -- leaf filter (in practice: node_level = max per subtree)
  GROUP BY b.node
)
SELECT leaf, leaf_revenue AS revenue, n_products FROM leaf_nodes
UNION ALL
SELECT 'Electronics', SUM(leaf_revenue), SUM(n_products) FROM leaf_nodes
ORDER BY revenue DESC
""").toPandas()
print("correct leaf-first rollup:")
print(fixed.to_string(index=False))

vals = dict(zip(fixed.leaf, fixed.revenue))
assert vals["Phones"] == 500.0 and vals["Laptops"] == 300.0 and vals["Tablets"] == 100.0
assert vals["Electronics"] == fact_total == 900.0
print("\nTalk track:")
print("* Each closure row carries the product's FULL revenue; leaf + ancestors = 2x (or Nx).")
print("* Fix options: (a) leaf-only filter via node_level/is_leaf, (b) pre-aggregate at leaf")
print("  grain and roll parents from leaf sums, (c) COUNT(DISTINCT product) for non-additive measures.")
print("* Faceted variants (a product in BOTH a style facet and a season facet) fan out rows even")
print("  further: aggregate-before-join, and never sum across facets.")
print("\nQ12 OK: naive 1800 vs true 900; leaf-first rollup makes every node reconcile to the fact")

## Q13 — Data Modeling — Semantic-Layer Governance with UC Metric Views `[FAANG]`

**Scenario (governance interview):** Two dashboards, two teams, two different "revenue" numbers — one computes `SUM(amount)`, the other nets out refunds — and neither matches Finance. 2025+ answer: define the metric **once** in a **Unity Catalog metric view** (governed, versioned, drillable by any dimension) instead of copy-pasting SQL into every dashboard.

**Requirements**

1. Show the divergence trap with two ad-hoc queries (gross 660 vs net 560 on the same table).
2. Create a metric view (`WITH METRICS ... LANGUAGE YAML`) with dimensions `region`, `channel` and measures `gross_revenue`, `net_revenue`, `order_count`, `refund_count` — `net_revenue` is the governed definition.
3. Query it with `MEASURE()` + `GROUP BY ALL` sliced by region, then by channel — the SAME governed definition serves every slice — and assert it matches the net ad-hoc number.

In [0]:
# Solution Q13 — one governed definition, drilled by any dimension
orders_tbl = "workspace.default.d29_q13_orders"
mv = "workspace.default.d29_q13_order_metrics"
spark.sql(f"DROP VIEW IF EXISTS {mv}")
spark.sql(f"DROP TABLE IF EXISTS {orders_tbl}")
spark.sql(f"""
CREATE TABLE {orders_tbl} (
  order_id STRING, region STRING, channel STRING, is_refund BOOLEAN, amount DOUBLE
)""")
spark.sql(f"""INSERT INTO {orders_tbl} VALUES
  ('o1','EMEA','web',false,120.0),('o2','EMEA','app',false,80.0),
  ('o3','APAC','web',false,150.0),('o4','APAC','app',true,50.0),
  ('o5','NA','web',false,200.0),('o6','NA','app',false,60.0)""")

# 1) the divergence trap: two ad-hoc dashboards, two different 'revenue' numbers
dash_a = spark.sql(f"SELECT SUM(amount) AS rev FROM {orders_tbl}").first()["rev"]
dash_b = spark.sql(f"""SELECT SUM(CASE WHEN is_refund THEN -amount ELSE amount END) AS rev
                       FROM {orders_tbl}""").first()["rev"]
print(f"ad-hoc dashboard A (gross): {dash_a} | ad-hoc dashboard B (net): {dash_b}")
assert dash_a == 660.0 and dash_b == 560.0 and dash_a != dash_b

# 2) governed definition in a UC metric view
spark.sql(f"""
CREATE VIEW {mv}
WITH METRICS
LANGUAGE YAML
AS $$
  version: 1.1
  source: >
    SELECT order_id, region, channel, is_refund, amount
    FROM workspace.default.d29_q13_orders
  dimensions:
    - name: region
      expr: region
    - name: channel
      expr: channel
  measures:
    - name: gross_revenue
      expr: SUM(amount)
    - name: net_revenue
      expr: SUM(CASE WHEN is_refund THEN -amount ELSE amount END)
    - name: order_count
      expr: COUNT(1)
    - name: refund_count
      expr: COUNT(1) FILTER (WHERE is_refund)
$$
""")

# 3) every team drills the SAME measure by its own dimensions
r1 = spark.sql(f"""
  SELECT `region`, MEASURE(`net_revenue`) AS `net_revenue`, MEASURE(`order_count`) AS `order_count`
  FROM {mv} GROUP BY ALL ORDER BY `net_revenue` DESC
""").toPandas()
print("by region:"); print(r1.to_string(index=False))
r2 = spark.sql(f"""
  SELECT `channel`, MEASURE(`net_revenue`) AS `net_revenue`, MEASURE(`refund_count`) AS `refund_count`
  FROM {mv} GROUP BY ALL ORDER BY `channel`
""").toPandas()
print("by channel:"); print(r2.to_string(index=False))

mv_rev = spark.sql(f"""SELECT MEASURE(`net_revenue`) AS `rev` FROM {mv} GROUP BY ALL""").first()["rev"]
assert mv_rev == dash_b == 560.0
assert float(r1[r1.region == "NA"].net_revenue.iloc[0]) == 260.0
assert int(r2[r2.channel == "app"].refund_count.iloc[0]) == 1
print("\nTalk track: metric views centralize the formula (refactor once, every dashboard fixes);")
print("they live in UC with ACLs + lineage; dashboard datasets import them directly; classic drift")
print("('is SUM(revenue) before or after refunds?') disappears.")
print("\nQ13 OK: 660-vs-560 divergence resolved by one governed net_revenue measure")

## Q14 — Snowflake — GEOGRAPHY vs GEOMETRY & Geofencing `[HARD]`

**Scenario (rideshare/delivery-platform interview):** Geofence 10M driver locations against pickup zones in Snowflake. The team stored lon/lat as `GEOMETRY` and computed Euclidean distances — numbers look plausible in the US, then explode near the **dateline** and at high latitude. Explain `GEOGRAPHY` (WGS84 spheroid, meters, geodesic) vs `GEOMETRY` (planar, SRID-dependent), and design the geofencing query with **Search Optimization**.

**Requirements**

1. Explain the two types and when each is right (projected coords/flat CAD → GEOMETRY; lon/lat anywhere on Earth → GEOGRAPHY).
2. **Prove the dateline catastrophe in Python**: two points 0.2° of longitude apart across ±180° — geodesic ≈ 22 km, naive planar-on-degrees ≈ 40,000 km. Also show the spheroid-vs-planar gap at mid-latitude.
3. Provide the reference DDL/queries: `GEOGRAPHY` column, `ST_DWITHIN` geofence, **Search Optimization Service** on the geo column for point-in-polygon pruning.

In [0]:
# Solution Q14 — geodesic vs planar, proven, then the reference SQL

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dphi, dlam = math.radians(lat2 - lat1), math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(p1)*math.cos(p2)*math.sin(dlam/2)**2
    return 2 * R * math.asin(math.sqrt(a))

# 1) the dateline catastrophe: 0.2 degrees of longitude apart, across +/-180
A = (0.0, -179.9); B = (0.0, 179.9)
geo = haversine_km(*A, *B)
planar_deg = math.hypot(A[0]-B[0], A[1]-B[1])          # Euclid on raw degrees
planar_km_naive = planar_deg * 111.32                    # '1 degree = 111 km' folklore
print(f"across the dateline: geodesic = {geo:.1f} km, planar-on-degrees = {planar_km_naive:,.0f} km")
assert geo < 100 and planar_km_naive > 30000

# 2) mid-latitude gap: SFO -> OAK, planar (no cos(lat) correction) vs geodesic
SFO = (37.6213, -122.3790); OAK = (37.7126, -122.2197)
g = haversine_km(*SFO, *OAK)
p = math.hypot(SFO[0]-OAK[0], SFO[1]-OAK[1]) * 111.32
print(f"SFO->OAK: geodesic = {g:.1f} km, naive planar = {p:.1f} km ({abs(p-g)/g*100:.0f}% error)")
assert 15 < g < 25 and p != g

print("\nReference SQL (Snowflake, not executed here):")
print("""
CREATE TABLE driver_locations (
  driver_id INT, ts TIMESTAMP_NTZ,
  loc GEOGRAPHY                  -- WGS84 spheroid, distances in METERS
);
-- ingest: ST_MAKEGEOGPOINT(lon, lat)   (argument order matters!)

-- geofence: drivers within 500 m of a pickup zone polygon
SELECT d.driver_id, z.zone_id
FROM driver_locations d
JOIN pickup_zones z
  ON ST_DWITHIN(d.loc, z.boundary, 500);   -- true geodesic meters

-- 10M-row geofence pruning: Search Optimization on the geo column
ALTER TABLE driver_locations ADD SEARCH OPTIMIZATION ON GEOGRAPHY(loc);
-- SOS builds a geo-cell index; ST_DWITHIN/ST_COVERS predicates prune
-- micro-partitions instead of scanning all 10M rows per refresh.
""")
print("Talk track: GEOMETRY assumes a flat plane (SRID 4326 lon/lat in GEOMETRY is a classic")
print("bug - degrees treated as Cartesian units); GEOGRAPHY computes on the spheroid.")
print("Near the dateline, antipodal handling flips: planar sees ~360 deg, geodesic ~22 km.")
print("\nQ14 OK: dateline 22 km vs 40,000 km proven; geofence + SOS design stated")

## Q15 — Snowflake — Cross-Cloud Replication & Failover Groups `[HARD]`

**Scenario (Netflix-style DR interview):** Primary Snowflake account in `us-east-1` replicates to `eu-west-1` via a **failover group** (replication + failover in one object). During a 30-minute regional outage you fail over; when the primary returns you must decide **when it is safe to fail back** — replication lag determines your data-loss (RPO) exposure. Explain the mechanics and model the lag.

**Requirements**

1. Explain failover groups vs replication groups: `ALLOW_WRITES` on the secondary during failover, scheduled refresh (`REPLICATION_SCHEDULE` in minutes), `SYSTEM$FAILOVER`... and the failback decision rule.
2. **Simulate the lag timeline**: writes arrive at 5,000 rows/min; refresh runs every 1 min (30s replication, 30s idle). During a 30-min outage the secondary accumulates local writes; on primary return, lag drains at (replication throughput − write rate). Compute when lag returns under the 5-minute RPO SLO and assert the failback gate opens.
3. State the operational talk track: zero-copy cloning for fast DR validation, `SHOW REPLICATION ACCOUNTS`, monitoring `REPLICATION_GROUP_REFRESH_HISTORY`.

In [0]:
# Solution Q15 — replication-lag timeline and the failback gate
WRITE_RATE = 5000        # rows/min at the primary
REPLICATE_RATE = 15000     # rows/min the secondary can replicate back (drain capacity)
REFRESH_INTERVAL = 1      # min
RPO_SLO_MIN = 5           # fail back only when lag < 5 min of writes

# normal ops: lag oscillates 0 -> REFRESH_INTERVAL minutes of writes
steady_lag_rows = WRITE_RATE * REFRESH_INTERVAL
print(f"steady-state lag: 0-{REFRESH_INTERVAL} min of writes "
      f"(up to {steady_lag_rows:,} rows) -> RPO satisfied")
assert steady_lag_rows / WRITE_RATE <= RPO_SLO_MIN

# outage: 30 min, secondary promoted (ALLOW_WRITES) and accumulates local writes
OUTAGE_MIN = 30
lag_rows = WRITE_RATE * OUTAGE_MIN          # un-replicated writes stranded on the secondary
print(f"after {OUTAGE_MIN} min outage: {lag_rows:,} rows of lag "
      f"({lag_rows/WRITE_RATE:.0f} min of writes)")
assert lag_rows == 150000

# failback gate: drain at (REPLICATE_RATE - WRITE_RATE) rows/min while primary
# keeps receiving new writes too
net_drain = REPLICATE_RATE - 2 * WRITE_RATE  # secondary replicates back AND keeps ingesting
print(f"net drain rate: {net_drain:,} rows/min")
assert net_drain > 0
drain_min = lag_rows / net_drain
gate_rows = RPO_SLO_MIN * WRITE_RATE
gate_min = (lag_rows - gate_rows) / net_drain
print(f"lag drops under the {RPO_SLO_MIN}-min RPO gate ({gate_rows:,} rows) after "
      f"{gate_min:.0f} min, fully drained after {drain_min:.0f} min")
assert gate_min < drain_min
print("\nReference SQL (Snowflake, not executed here):")
print("""
-- one object = replication + failover, whole-account scope:
CREATE FAILOVER GROUP my_fg
  FROM ACCOUNTS us_east_1 TO ACCOUNTS eu_west_1
  ALLOW_WRITES                       -- secondary becomes writable when promoted
  REPLICATION_SCHEDULE = '1 MINUTE';

-- DR drill / real outage on the secondary account:
ALTER FAILOVER GROUP my_fg PRIMARY;      -- (run on secondary) SYSTEM$FAILOVER flow

-- failback decision: inspect refresh history first
SELECT replication_time_lag FROM TABLE(
  INFORMATION_SCHEMA.REPLICATION_GROUP_REFRESH_HISTORY(
    GROUP_NAME => 'MY_FG'))
ORDER BY refresh_start_time DESC LIMIT 5;

-- fast DR validation without copying data: zero-copy clone on the secondary
CREATE TABLE orders_dr_validation CLONE orders;
""")
print("Talk track: during failover DO NOT reverse the group until lag < RPO SLO, else you")
print("re-diverge; if both sides took writes you need a conflict-resolution pass (bi-directional")
print("replication is the hard interview follow-up: last-writer-wins vs merge tables).")
print("\nQ15 OK: 150k-row lag modeled; failback gate opens only under the 5-min RPO")

## Q16 — Monitoring — Multi-Window Multi-Burn-Rate SLO Alerting `[FAANG]`

**Scenario (Google-SRE-style on-call interview):** A service has a 99.9% availability SLO over 30 days → a **0.1% error budget** (43.2 error-minutes per 30 days — 0.1% of 43,200 minutes). Single-window alerting either pages on every blip or misses real budget exhaustion. Design the canonical **multi-window multi-burn-rate** alerting and demonstrate it on a simulated metric series.

**Requirements**

1. Define burn rate = error rate / SLO error budget, and the budget math (43.2 error-minutes; a 2%-error hour burns ~2.8% of the monthly budget).
2. Canonical alert matrix: **PAGE** on ≥14.4x over BOTH a 1h and a 5m window (fast burn); **PAGE** on ≥6x over BOTH 6h and 30m; **TICKET** on ≥3x over 1d (+2h confirmation); **TICKET** on ≥1x over 3d (+6h). Explain why the long window gates the short one (blip immunity).
3. Simulate 48h of hourly error rates — baseline 0.02%, a 2-hour incident at 2% — and prove: exactly the fast-burn PAGE fires, the slow tiers stay quiet, and a baseline hour (0.2x burn) never alerts.

In [0]:
# Solution Q16 — budget math + the canonical alert matrix, demonstrated
SLO_ERROR_RATE = 0.001                       # 99.9% SLO
BUDGET_MINUTES = 30 * 24 * 60 * SLO_ERROR_RATE
print(f"error budget: {BUDGET_MINUTES:.0f} error-minutes per 30d "
      f"(0.1% of 43,200 min)")

# metric simulation: 48 hourly points; incident = 2% error at hours 23-24
err_rates = [0.02 if h in (23, 24) else 0.0002 for h in range(48)]

def burn(window_rates):
    return (sum(window_rates) / len(window_rates)) / SLO_ERROR_RATE

# window views around the incident peak (hour 23)
fast_1h = burn(err_rates[23:24])                       # the incident hour
fast_5m = burn([0.02] * 12)                           # 12 five-minute samples in that hour
six_h   = burn(err_rates[18:24])                       # 5 baseline + 1 incident hours
thirty_min = burn([0.0002] * 25 + [0.02] * 5)          # incident started 30 min ago
day_w   = burn(err_rates)                              # full 48h
print(f"burn rates: 1h={fast_1h:.1f}x | 5m={fast_5m:.1f}x | 6h={six_h:.2f}x | "
      f"30m={thirty_min:.2f}x | 48h={day_w:.2f}x")

page_fast  = fast_1h >= 14.4 and fast_5m >= 14.4
page_slow  = six_h >= 6 and thirty_min >= 6
ticket_1d  = day_w >= 3
print(f"fast-burn PAGE (14.4x on 1h AND 5m): {page_fast}")
print(f"slow-burn PAGE (6x on 6h AND 30m):   {page_slow}")
print(f"slow TICKET (3x on 1d + 2h confirm): {ticket_1d}")
assert page_fast and not page_slow and not ticket_1d
assert 3.0 < six_h < 6.0        # derived: (5*0.0002 + 0.02)/6/0.001 = 3.5x

# budget arithmetic for the talk track
incident_err_min = 2 * 60 * 0.02
print(f"the 2h incident burned {incident_err_min:.0f} error-minutes = "
      f"{incident_err_min/BUDGET_MINUTES*100:.1f}% of the 30d budget")
assert incident_err_min == 2.4

# baseline hour: 0.2x burn — the long window is what keeps every blip quiet
baseline = burn(err_rates[0:1])
print(f"baseline hour burn: {baseline:.1f}x -> no alert, ever")
assert baseline == 0.2
print("\nTalk track: the SHORT window catches fast budget burn; the LONG window confirms")
print("it's sustained (blip immunity); a single 20-min degradation at 20x burn pages only if")
print("the 1h window also clears 14.4x — that's the whole design. In Databricks SQL land this")
print("maps to a metrics-table pattern: hourly error rates into a table, SQL alerts on top.")
print("\nQ16 OK: exactly one tier (fast-burn PAGE) fires; baseline and slow tiers stay quiet")

## Q17 — Monitoring — Multi-Root-Cause Triage `[HARD]`

**Scenario (Lakeflow-Jobs incident interview):** You wake up to a pipeline page: 8 nodes down. The job graph shows TWO independent source failures at the same timestamp — `bronze_orders` and `bronze_events` — and downstream nodes that depend on BOTH (shared victims). A naive "blame the most recent failure" heuristic misdiagnoses. Build the triage algorithm.

**Requirements**

1. Model the DAG and statuses; find ALL **failed roots** — a root is a FAILED node with no FAILED ancestor (its failure is not explained by anything upstream).
2. Classify victims (SKIPPED/STALE) and attribute each victim to the root(s) that explain it — shared victims (downstream of both roots) must be attributed to BOTH, not double-billed as a third cause.
3. Assert: 2 roots found, bronze_events explains 5 victims, bronze_orders 3, and gold_orders + dashboard are shared.

In [0]:
# Solution Q17 — root isolation on a multi-failure DAG
edges = {
    "bronze_orders": [],   "bronze_events": [],
    "silver_orders": ["bronze_orders"], "silver_events": ["bronze_events"],
    "dim_customers": ["bronze_events"],                  # same source, different path
    "gold_orders":  ["silver_orders", "dim_customers"], # child of BOTH roots' lineages
    "gold_events":  ["silver_events"],
    "dashboard":    ["gold_orders", "gold_events"],
}
statuses = {
    "bronze_orders": "FAILED", "bronze_events": "FAILED",     # simultaneous, independent
    "silver_orders": "SKIPPED", "silver_events": "SKIPPED",
    "dim_customers": "SKIPPED", "gold_orders": "SKIPPED",
    "gold_events": "SKIPPED", "dashboard": "STALE",
}

def ancestors(n, dg):
    stack, seen = list(dg[n]), set()
    while stack:
        a = stack.pop()
        if a in seen: continue
        seen.add(a); stack.extend(dg[a])
    return seen

def find_roots(nodes, st, dg):
    """A root = FAILED with no FAILED ancestor (failure unexplained upstream)."""
    return sorted(n for n in nodes
                  if st[n] == "FAILED"
                  and not any(st[a] == "FAILED" for a in ancestors(n, dg)))

roots_found = find_roots(statuses.keys(), statuses, edges)
print("failed roots:", roots_found)
assert roots_found == ["bronze_events", "bronze_orders"]

victims = sorted(n for n in statuses if statuses[n] in ("SKIPPED", "STALE"))
impact = {r: sum(1 for n in victims if r in ancestors(n, edges)) for r in roots_found}
print("victims:", victims)
print("victims explained by each root:", impact)
assert impact == {"bronze_events": 5, "bronze_orders": 3}

shared = [n for n in victims if all(r in ancestors(n, edges) for r in roots_found)]
print("shared victims (downstream of BOTH roots):", sorted(shared))
assert sorted(shared) == ["dashboard", "gold_orders"]
print("\nTalk track:")
print("* 'Most recent failure' mis-blames here: the two roots share a timeline; only lineage")
print("  traversal separates them.")
print("* Shared victims are why victim-count != root-severity: gold_orders fails for BOTH")
print("  causes; fix either root and it stays down until both are healthy.")
print("* In Databricks: query system.lakeflow.job_run_timeline / task lifecycle events for")
print("  FAILED roots, then walk downstream deps; repair-run BOTH roots, not one.")
print("\nQ17 OK: 2 independent roots isolated; 5+3 victim attribution with shared victims explained")

## Q18 — Monitoring — Predictive SLA ETA with a Regression Forecast `[FAANG]`

**Scenario (Meta/Uber SLA-observability interview):** A nightly job has a 160-minute SLA, and its runtime is **drifting upward** (+1.5 min/day from data growth) plus a **+18 min weekend effect**. Retrospective "job breached at 09:00" alerts are too late — build a **pre-breach predictor**: regress historical run durations on (day-index, weekend-flag), forecast tonight's run with a **p90 upper band**, and fire the alert *before* the run starts.

**Requirements**

1. Generate 61 days of history: `dur = 50 + 1.5·day + 18·weekend + N(0, 3)`; fit **two-feature OLS via normal equations** (no sklearn) solving a 3×3 system by Gaussian elimination.
2. Report the recovered coefficients (they must be close to the true 1.5 / 18), RMSE, R², the next-run point prediction (day 61 is a weekend day), and the p90 upper bound `pred + 1.645·RMSE`.
3. Assert the model recovers the true structure and the **pre-breach alert fires** (p90 ETA > 160-min SLA) *before* the job has run.

In [0]:
# Solution Q18 — normal-equations OLS + p90 pre-breach gate
random.seed(29)
N_DAYS = 61
hist, feats = [], []
for i in range(N_DAYS):
    weekend = 1 if (i % 7) in (5, 6) else 0
    hist.append(50 + 1.5 * i + 18 * weekend + random.gauss(0, 3))   # truth with noise
    feats.append((1.0, float(i), float(weekend)))                   # design matrix rows

def solve3(A, b):
    """3x3 Gaussian elimination with partial pivoting."""
    M = [row[:] + [b[i]] for i, row in enumerate(A)]
    for c in range(3):
        p = max(range(c, 3), key=lambda r: abs(M[r][c]))
        M[c], M[p] = M[p], M[c]
        for r in range(3):
            if r != c:
                f = M[r][c] / M[c][c]
                M[r] = [M[r][k] - f * M[c][k] for k in range(4)]
    return [M[i][3] / M[i][i] for i in range(3)]

XtX = [[sum(f[a] * f[b] for f in feats) for b in range(3)] for a in range(3)]
Xty = [sum(f[a] * y for f, y in zip(feats, hist)) for a in range(3)]
beta = solve3(XtX, Xty)                       # [intercept, day_coef, weekend_coef]
pred = [beta[0] + beta[1] * f[1] + beta[2] * f[2] for f in feats]
resid = [y - p for y, p in zip(hist, pred)]
rmse = math.sqrt(sum(r * r for r in resid) / len(resid))
ss_tot = sum((y - sum(hist) / len(hist)) ** 2 for y in hist)
r2 = 1 - sum(r * r for r in resid) / ss_tot
p90 = 1.645 * rmse

print(f"fit: dur = {beta[0]:.1f} + {beta[1]:.2f}*day + {beta[2]:.1f}*weekend")
print(f"     (truth: 50 + 1.50*day + 18*weekend) | RMSE {rmse:.2f} min | R^2 {r2:.3f}")
assert r2 > 0.97
assert abs(beta[1] - 1.5) < 0.1 and abs(beta[2] - 18) < 3   # structure recovered

next_weekend = 1 if (N_DAYS % 7) in (5, 6) else 0
pred_next = beta[0] + beta[1] * N_DAYS + beta[2] * next_weekend
eta_p90 = pred_next + p90
SLA_MIN = 160
print(f"next run: day {N_DAYS}, weekend={next_weekend} -> point {pred_next:.1f} min, "
      f"p90 band {eta_p90:.1f} min, SLA {SLA_MIN} min")
print(f"PRE-BREACH ALERT (fires BEFORE the run): {eta_p90 > SLA_MIN}")
assert 140 < pred_next < 170 and eta_p90 > SLA_MIN
print("\nTalk track:")
print("* Point prediction alone under-alerts: the p90 band prices in residual uncertainty.")
print("* Fit drift as a FEATURE (day index) so the alert threshold self-tightens as data grows.")
print("* At scale: log run durations to a metrics table (system.lakeflow job-run history),")
print("  retrain nightly, and expose ETA = now + p90(pred) as a SQL alert condition.")
print("* Watch collinearity: if weekend flag correlates with day index over short windows,")
print("  ridge-regularize or widen the training window.")
print("\nQ18 OK: coefficients recovered, pre-breach alert fires with p90 margin before the run")

---
# 📋 Day 29 Scorecard — FAANG / Top Product Company

Score yourself per question: **0** couldn't attempt · **1** needed the solution · **2** got it with hints · **3** solved cold. Anything < 3 goes in the review log.

| # | Topic | Tag | Score (0-3) | Key gap to review |
|---|---|---|---|---|
| 1 | Item-item CF in SQL (full-vector cosine) | [FAANG] |  | co-rated norms trap |
| 2 | Kaplan-Meier survival in SQL | [FAANG] |  | at-risk set / censoring |
| 3 | Aho-Corasick automaton | [FAANG] |  | fail/output links |
| 4 | Hopcroft-Karp matching | [FAANG] |  | layered BFS/DFS |
| 5 | applyInPandasWithState | [FAANG] |  | state.exists / tuple values |
| 6 | pandas-on-Spark traps | [HARD] |  | index is not a column |
| 7 | Speculative execution | [HARD] |  | first-committer-wins / Delta OCC |
| 8 | Spark Connect internals | [HARD] |  | plan-shipping, no JVM |
| 9 | Dual-stream backfill | [HARD] |  | one checkpoint per stream |
| 10 | LLM corpus prep | [FAANG] |  | shingle Jaccard / leak-proof splits |
| 11 | Kafka rebalance & replay | [FAANG] |  | at-least-once + idempotent sink |
| 12 | Taxonomy bridge double-count | [HARD] |  | leaf-first rollup |
| 13 | UC metric view governance | [FAANG] |  | MEASURE() / GROUP BY ALL |
| 14 | GEOGRAPHY vs GEOMETRY | [HARD] |  | dateline wrap / ST_DWITHIN |
| 15 | Snowflake failover groups | [HARD] |  | lag-based failback gate |
| 16 | Multi-window burn-rate SLOs | [FAANG] |  | 14.4x / long-window gating |
| 17 | Multi-root triage | [HARD] |  | roots vs shared victims |
| 18 | Predictive SLA ETA | [FAANG] |  | p90 band / normal equations |

**Debrief prompts**

* Which three questions took longest? Was it recall (didn't know the API/algorithm) or synthesis (knew the parts, couldn't combine)?
* For every [HARD] you scored < 3: re-derive the *why* (why full-vector norms, why leaf-first rollup, why one checkpoint per stream) — FAANG follow-ups always probe the why.
* The theme of this week's questions: **algorithms you'd normally hand to a library** (CF, KM, Aho-Corasick, Hopcroft-Karp) and **surfaces where the docs are thin** (pandas-on-Spark, Spark Connect, metric views, GroupState). Practice articulating the trade-offs out loud.

**Next**: Day 30 — round-number day; expect a mixed re-quiz of this week's weakest topics plus system-design synthesis questions.